# Test Rebalancer

End-to-end test of rebalancing pipeline on top of `DataLoaderGraph` snapshots.

In [ ]:
import pandas as pd
from IPython.display import display

from gbp.loaders import DataLoaderGraph, DataLoaderMock, GraphLoaderConfig
from gbp.rebalancer import DataLoaderRebalancer, Rebalancer, RebalancerConfig

## 1. Load graph data

In [ ]:
from gbp.build.pipeline import build_model

mock = DataLoaderMock({"n_stations": 8, "n_depots": 2, "n_timestamps": 48})
loader = DataLoaderGraph(mock, GraphLoaderConfig(distance_backend="haversine"))
raw = loader.load()
resolved = build_model(raw)

print(f"Available dates: {len(loader.available_dates)}")
print(loader.available_dates[:3])

print(f"Telemetry rows: {len(mock.df_telemetry_ts)}")
print(f"Trips rows: {len(mock.df_trips)}")
print(f"Station costs rows: {len(mock.df_station_costs)}")
print(f"Truck rates rows: {len(mock.df_truck_rates)}")

display(mock.df_telemetry_ts.head(3))

In [ ]:
print(f"Resolved facilities: {len(resolved.facilities)}")
print(f"Resolved edges: {len(resolved.edges) if resolved.edges is not None else 0}")
print(f"Full inventory_ts shape: {loader.source.df_inventory_ts.shape}")
print(f"Full telemetry_ts rows: {len(loader.source.df_telemetry_ts)}")
print(f"Available dates: {len(loader.available_dates)}")

display(loader.source.df_inventory_ts.head(3))
display(loader.source.df_telemetry_ts.head(3))

In [4]:
preview_date = loader.available_dates[0]
snapshot = loader.rebalancer_snapshot(preview_date)
_tel = loader.source.df_telemetry_ts
_metrics = [c for c in ("num_bikes_available", "num_ebikes_available", "num_docks_available") if c in _tel.columns]
tel_long = _tel[["timestamp", "station_id"] + _metrics].melt(
    id_vars=["timestamp", "station_id"], var_name="metric", value_name="value"
).rename(columns={"station_id": "node_id"})
tel_at = tel_long[tel_long["timestamp"] == preview_date]

print(f"Preview snapshot date: {preview_date}")
print(f"Snapshot inventory rows: {len(snapshot.inventory)}")
print(f"Telemetry long rows at timestamp: {len(tel_at)}")

display(snapshot.inventory.head(5))
display(tel_at.head(5))

Preview snapshot date: 2025-01-01 00:00:00
Snapshot inventory rows: 8
Snapshot telemetry rows: 64


,node_id,commodity_id,quantity
0,station_1,bike,5
1,station_2,bike,34
2,station_3,bike,29
3,station_4,bike,4
4,station_5,bike,15


,node_id,timestamp,metric,value
0,station_1,2025-01-01,num_bikes_available,5
1,station_2,2025-01-01,num_bikes_available,34
2,station_3,2025-01-01,num_bikes_available,29
3,station_4,2025-01-01,num_bikes_available,4
4,station_5,2025-01-01,num_bikes_available,15


## 2. Configure rebalancer

In [5]:
config = RebalancerConfig(
    inventory_node_type="station",
    depot_node_type="depot",
    min_threshold=0.3,
    max_threshold=0.7,
    time_limit_seconds=5,
)

dataloader_rebalancer = DataLoaderRebalancer(loader, config)
rebalancer = Rebalancer(dataloader_rebalancer, config)
config

RebalancerConfig(inventory_node_type='station', depot_node_type='depot', min_threshold=0.3, max_threshold=0.7, time_limit_seconds=5)

## 3. Find a snapshot with imbalance

In [6]:
selected_date = None

for d in loader.available_dates:
    dataloader_rebalancer.load_data(date=d)
    if dataloader_rebalancer.data is not None:
        selected_date = d
        break

if selected_date is None:
    print("No imbalanced snapshot was found in current mock data.")
else:
    print(f"Selected snapshot: {selected_date}")
    print(f"Sources: {(dataloader_rebalancer.df_node_demand['utilization'] > config.max_threshold).sum()}")
    print(f"Destinations: {(dataloader_rebalancer.df_node_demand['utilization'] < config.min_threshold).sum()}")

2026-03-09 20:43:06 [debug    ] snapshot                       date='2025-01-01 00:00:00' loader=graph
Selected snapshot: 2025-01-01 00:00:00
Sources: 2
Destinations: 4


## 4. Inspect node demand table

In [7]:
df_node_demand = dataloader_rebalancer.df_node_demand.copy()
display(df_node_demand.sort_values("utilization", ascending=False))

,node_id,latitude,longitude,inventory_capacity,quantity,utilization,target_count,balance
4,station_5,40.707772,-74.013110,18,15,0.833333,9.0,6.0
6,station_7,40.814057,-74.000502,35,28,0.800000,17.5,10.5
1,station_2,40.715035,-73.981840,49,34,0.693878,24.5,9.5
2,station_3,40.764010,-73.968958,46,29,0.630435,23.0,6.0
0,station_1,40.820109,-73.987643,22,5,0.227273,11.0,-6.0
5,station_6,40.802949,-73.968158,58,11,0.189655,29.0,-18.0
3,station_4,40.687885,-74.005369,30,4,0.133333,15.0,-11.0
7,station_8,40.854152,-73.942924,55,0,0.000000,27.5,-27.5


## 5. Inspect generated PDP model

In [8]:
if dataloader_rebalancer.data is None:
    print("No PDP model: current snapshot has no source/destination pairs.")
else:
    pdp = dataloader_rebalancer.data
    print(f"num_resources={pdp['num_resources']}")
    print(f"num_pairs={len(pdp['pairs'])}")
    print(f"matrix_shape={pdp['distance_matrix'].shape}")
    print(f"node_layout_example={pdp['node_ids'][:6]}")
    display(pd.DataFrame(pdp['pairs']).head())

num_resources=3
num_pairs=2
matrix_shape=(5, 5)
node_layout_example=['depot', 'station_7_pickup', 'station_8_delivery', 'station_5_pickup', 'station_8_delivery']


,pickup_node_id,pickup_latitude,pickup_longitude,delivery_node_id,delivery_latitude,delivery_longitude,quantity
0,station_7,40.814057,-74.000502,station_8,40.854152,-73.942924,10
1,station_5,40.707772,-74.013110,station_8,40.854152,-73.942924,6


## 6. Run full rebalancing pipeline

In [9]:
if selected_date is None:
    print("Pipeline not executed because no imbalanced snapshot was found.")
else:
    rebalancer.run(date=selected_date)

2026-03-09 20:43:10 [debug    ] snapshot                       date='2025-01-01 00:00:00' loader=graph
Solution found — objective: 35791, total distance: 35791, dropped nodes: []


## 7. Route output

In [10]:
if hasattr(rebalancer, "route_df") and rebalancer.route_df is not None:
    display(rebalancer.route_df)
else:
    print("No route output (no feasible solution or no imbalances).")

,resource_id,step,node_id,action,quantity,cumulative_load
0,0,0,depot,depot,0,0
1,0,1,station_7,pickup,10,10
2,0,2,station_5,pickup,6,16
3,0,3,station_8,delivery,6,10
4,0,4,station_8,delivery,10,0
5,0,5,depot,depot,0,0


## 8. Inventory before/after

In [11]:
if hasattr(rebalancer, "df_updated") and rebalancer.df_updated is not None:
    cols = [
        "node_id",
        "inventory_capacity",
        "old_quantity",
        "quantity",
        "inventory_change",
        "utilization",
        "new_utilization",
    ]
    display(rebalancer.df_updated[cols].sort_values("inventory_change", ascending=False))
else:
    print("No updated inventory output.")

,node_id,inventory_capacity,old_quantity,quantity,inventory_change,utilization,new_utilization
7,station_8,55,0,16,16,0.000000,0.290909
0,station_1,22,5,5,0,0.227273,0.227273
2,station_3,46,29,29,0,0.630435,0.630435
1,station_2,49,34,34,0,0.693878,0.693878
3,station_4,30,4,4,0,0.133333,0.133333
5,station_6,58,11,11,0,0.189655,0.189655
4,station_5,18,15,9,-6,0.833333,0.500000
6,station_7,35,28,18,-10,0.800000,0.514286


## 9. How many snapshots are rebalanceable?

In [12]:
stats = []

for d in loader.available_dates:
    dataloader_rebalancer.load_data(date=d)
    stats.append({
        "date": d,
        "has_pdp_model": dataloader_rebalancer.data is not None,
    })

df_stats = pd.DataFrame(stats)
print(df_stats["has_pdp_model"].value_counts())
display(df_stats.head(10))

2026-03-09 20:43:22 [debug    ] snapshot                       date='2025-01-01 00:00:00' loader=graph
2026-03-09 20:43:22 [debug    ] snapshot                       date='2025-01-01 01:00:00' loader=graph
2026-03-09 20:43:22 [debug    ] snapshot                       date='2025-01-01 02:00:00' loader=graph
2026-03-09 20:43:22 [debug    ] snapshot                       date='2025-01-01 03:00:00' loader=graph
2026-03-09 20:43:22 [debug    ] snapshot                       date='2025-01-01 04:00:00' loader=graph
2026-03-09 20:43:22 [debug    ] snapshot                       date='2025-01-01 05:00:00' loader=graph
2026-03-09 20:43:22 [debug    ] snapshot                       date='2025-01-01 06:00:00' loader=graph
2026-03-09 20:43:23 [debug    ] snapshot                       date='2025-01-01 07:00:00' loader=graph
2026-03-09 20:43:23 [debug    ] snapshot                       date='2025-01-01 08:00:00' loader=graph
2026-03-09 20:43:23 [debug    ] snapshot                       date='2025

,date,has_pdp_model
0,2025-01-01 00:00:00,True
1,2025-01-01 01:00:00,True
2,2025-01-01 02:00:00,True
3,2025-01-01 03:00:00,True
4,2025-01-01 04:00:00,True
5,2025-01-01 05:00:00,True
6,2025-01-01 06:00:00,True
7,2025-01-01 07:00:00,True
8,2025-01-01 08:00:00,True
9,2025-01-01 09:00:00,True
